# ROGII — Wellbore Geology Prediction
## Notebook 2: Feature Engineering + Modeling + Submission

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
from data_utils import *

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import pywt

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_DIR   = Path('../data')
SUBMIT_DIR = Path('../submissions')
SUBMIT_DIR.mkdir(exist_ok=True)

SEED    = 42
N_FOLDS = 5

## 1. Load Data

In [2]:
train = load_all_wells(DATA_DIR, split='train')
test  = load_all_wells(DATA_DIR, split='test')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {train[WELL_COL].nunique()} wells, {len(train):,} rows')
print(f'Test : {test[WELL_COL].nunique()} wells, {len(test):,} rows')
print(f'Submission rows: {len(sample_sub)}')

Loaded train: 773 wells, 5,092,255 rows, 16 columns
Loaded test: 3 wells, 19,221 rows, 9 columns
Train: 773 wells, 5,092,255 rows
Test : 3 wells, 19,221 rows
Submission rows: 14151


## 2. Feature Engineering

**Only features available in both train and test** are safe to use as model inputs:
`MD, X, Y, Z, GR, TVT_input, tw_GR` (typewell GR aligned by TVT_input)

In [3]:
def compute_tortuosity(df, window=10):
    dx = df[X_COL].diff().fillna(0)
    dy = df[Y_COL].diff().fillna(0)
    dz = df[Z_COL].diff().fillna(0)
    ds    = np.sqrt(dx**2 + dy**2 + dz**2)
    arc   = ds.rolling(window, min_periods=1).sum()
    chord = np.sqrt(
        (df[X_COL] - df[X_COL].shift(window)).fillna(0)**2 +
        (df[Y_COL] - df[Y_COL].shift(window)).fillna(0)**2 +
        (df[Z_COL] - df[Z_COL].shift(window)).fillna(0)**2
    )
    return (arc / chord.replace(0, np.nan)).fillna(1.0)


def dwt_features(series, wavelet='db4', level=3):
    coeffs = pywt.wavedec(series.fillna(series.median()).values, wavelet, level=level)
    out = {}
    for i, c in enumerate(coeffs):
        up = np.interp(np.linspace(0, len(c)-1, len(series)), np.arange(len(c)), c)
        label = 'A' if i == 0 else f'D{i}'
        out[f'dwt_{label}'] = up
    return pd.DataFrame(out, index=series.index)


def engineer_features(df):
    """
    Build features per-well to avoid leakage across wells.
    Only uses columns available in both train and test.
    """
    df = df.copy()

    # Per-well GR normalisation
    stats = df.groupby(WELL_COL)[GR_COL].agg(['mean', 'std']).rename(
        columns={'mean': '_mu', 'std': '_sd'})
    df = df.merge(stats, on=WELL_COL, how='left')
    df['GR_norm'] = (df[GR_COL] - df['_mu']) / (df['_sd'] + 1e-6)
    df.drop(columns=['_mu', '_sd'], inplace=True)

    # Typewell GR normalisation (same idea)
    if 'tw_GR' in df.columns:
        tw_stats = df.groupby(WELL_COL)['tw_GR'].agg(['mean', 'std']).rename(
            columns={'mean': '_mu', 'std': '_sd'})
        df = df.merge(tw_stats, on=WELL_COL, how='left')
        df['tw_GR_norm'] = (df['tw_GR'] - df['_mu']) / (df['_sd'] + 1e-6)
        df.drop(columns=['_mu', '_sd'], inplace=True)
        df['GR_vs_tw'] = df['GR_norm'] - df['tw_GR_norm']   # horizontal GR deviation from typewell

    parts = []
    for _, gdf in df.groupby(WELL_COL):
        gdf = gdf.sort_values(DEPTH_COL).copy()
        gr  = gdf['GR_norm']

        # ── Rolling statistics ──────────────────────────────────────────
        for w in [3, 5, 10, 20, 50]:
            r = gr.rolling(w, min_periods=1)
            gdf[f'gr_mean_{w}']  = r.mean()
            gdf[f'gr_std_{w}']   = r.std().fillna(0)
            gdf[f'gr_range_{w}'] = r.max() - r.min()
            gdf[f'gr_skew_{w}']  = r.skew().fillna(0)

        # ── EMA + MACD ──────────────────────────────────────────────────
        for span in [5, 10, 20, 50]:
            gdf[f'ema_{span}'] = gr.ewm(span=span, adjust=False).mean()
        for fast, slow in [(5, 20), (10, 50)]:
            macd = gdf[f'ema_{fast}'] - gdf[f'ema_{slow}']
            gdf[f'macd_{fast}_{slow}']      = macd
            gdf[f'macd_hist_{fast}_{slow}'] = macd - macd.ewm(span=9, adjust=False).mean()

        # ── Lags & leads ────────────────────────────────────────────────
        for lag in [1, 2, 3, 5, 10, 15, 20, 30]:
            gdf[f'gr_lag_{lag}']  = gr.shift(lag)
            gdf[f'gr_lead_{lag}'] = gr.shift(-lag)

        # ── Derivatives ─────────────────────────────────────────────────
        gdf['gr_d1']     = gr.diff(1).fillna(0)
        gdf['gr_d2']     = gr.diff(2).fillna(0)
        gdf['gr_d1_abs'] = gdf['gr_d1'].abs()

        # ── TVT_input features ──────────────────────────────────────────
        # Where TVT_input is known (non-zero), carry it forward/backward
        tvt_in = gdf[TVTINPUT_COL].replace(0, np.nan)
        gdf['tvtin_ffill'] = tvt_in.ffill()   # last known TVT
        gdf['tvtin_bfill'] = tvt_in.bfill()   # next known TVT
        gdf['tvtin_interp'] = tvt_in.interpolate(method='linear')  # linear interpolation
        gdf['tvtin_known']  = (gdf[TVTINPUT_COL] != 0).astype(float)
        gdf['tvtin_dist_fwd'] = (~tvt_in.isna()).cumsum()  # distance from last anchor

        # ── DWT on GR ───────────────────────────────────────────────────
        try:
            dwt = dwt_features(gr, wavelet='db4', level=3)
            for col in dwt.columns:
                gdf[col] = dwt[col].values
        except Exception:
            pass

        # ── 3D trajectory ────────────────────────────────────────────────
        for w in [5, 10, 20]:
            gdf[f'tort_{w}'] = compute_tortuosity(gdf, window=w)
        dx = gdf[X_COL].diff().fillna(0)
        dy = gdf[Y_COL].diff().fillna(0)
        dz = gdf[Z_COL].diff().fillna(0)
        gdf['dip'] = np.degrees(np.arctan2(dz.abs(),
                                np.sqrt(dx**2 + dy**2) + 1e-6))
        gdf['dip_d1'] = gdf['dip'].diff().fillna(0)

        # ── Depth / position ─────────────────────────────────────────────
        dep_range = gdf[DEPTH_COL].max() - gdf[DEPTH_COL].min() + 1e-6
        gdf['depth_norm'] = (gdf[DEPTH_COL] - gdf[DEPTH_COL].min()) / dep_range
        gdf['well_pos']   = np.arange(len(gdf)) / len(gdf)
        gdf['n_samples']  = len(gdf)

        parts.append(gdf)

    return pd.concat(parts, ignore_index=True)


print('Engineering features...')
train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')

Engineering features...
Train: (5092255, 79)  |  Test: (19221, 72)


## 3. Prepare Feature Matrix

In [4]:
drop_cols = [TARGET_COL, WELL_COL, 'row_idx', GR_COL, 'tw_GR',
             'tw_TVT', 'tw_Geology', 'sub_id', 'pred']
feature_cols = [
    c for c in train_fe.columns
    if c not in drop_cols and train_fe[c].dtype != object
]

X_train = train_fe[feature_cols].astype(np.float32)
y_train = train_fe[TARGET_COL].astype(np.float32)
groups  = train_fe[WELL_COL]

# reindex: test lacks resistivity columns → they appear as NaN, filled below
X_test  = test_fe.reindex(columns=feature_cols).astype(np.float32)

medians = X_train.median()
X_train = X_train.fillna(medians)
X_test  = X_test.fillna(medians)

gkf    = GroupKFold(n_splits=N_FOLDS)
splits = list(gkf.split(X_train, y_train, groups=groups))

print(f'Feature matrix: {X_train.shape}')
print(f'Test matrix   : {X_test.shape}')
print(f'Features ({len(feature_cols)}): {feature_cols[:8]}...')

for fold_i, (tr, val) in enumerate(splits):
    print(f'  Fold {fold_i}: train {len(tr):6,} rows '
          f'({groups.iloc[tr].nunique()} wells)  '
          f'val {len(val):6,} rows '
          f'({groups.iloc[val].nunique()} wells)')

Feature matrix: (5092255, 74)
Test matrix   : (19221, 74)
Features (74): ['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU']...
  Fold 0: train 4,075,197 rows (619 wells)  val 1,017,058 rows (154 wells)
  Fold 1: train 4,075,198 rows (619 wells)  val 1,017,057 rows (154 wells)
  Fold 2: train 4,073,152 rows (618 wells)  val 1,019,103 rows (155 wells)
  Fold 3: train 4,072,802 rows (618 wells)  val 1,019,453 rows (155 wells)
  Fold 4: train 4,072,671 rows (618 wells)  val 1,019,584 rows (155 wells)


## 4. LightGBM

In [ ]:
lgb_params = {
    'objective':          'regression',
    'metric':             'rmse',
    'learning_rate':      0.1,      # higher LR → fewer rounds needed
    'num_leaves':         127,
    'max_depth':          -1,
    'min_child_samples':  20,
    'feature_fraction':   0.8,
    'bagging_fraction':   0.8,
    'bagging_freq':       1,
    'reg_alpha':          0.1,
    'reg_lambda':         0.5,
    'n_jobs':             -1,
    'verbosity':          -1,
    'seed':               SEED,
}

lgb_oof   = np.zeros(len(X_train))
lgb_test  = np.zeros(len(X_test))

for fold_i, (tr, val) in enumerate(splits):
    dtrain = lgb.Dataset(X_train.iloc[tr], y_train.iloc[tr])
    dval   = lgb.Dataset(X_train.iloc[val], y_train.iloc[val], reference=dtrain)
    m = lgb.train(
        lgb_params, dtrain, num_boost_round=500,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(100)]
    )
    lgb_oof[val]  = m.predict(X_train.iloc[val])
    lgb_test     += m.predict(X_test) / N_FOLDS
    fold_rmse = np.sqrt(mean_squared_error(y_train.iloc[val], lgb_oof[val]))
    print(f'  Fold {fold_i}  RMSE: {fold_rmse:.4f}  best_iter={m.best_iteration}')

lgb_rmse = np.sqrt(mean_squared_error(y_train, lgb_oof))
print(f'\nLightGBM OOF RMSE: {lgb_rmse:.4f}')

## 5. XGBoost

In [ ]:
xgb_params = dict(
    objective='reg:squarederror', eval_metric='rmse',
    learning_rate=0.1, max_depth=6, min_child_weight=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    n_estimators=500, early_stopping_rounds=30,
    random_state=SEED, n_jobs=-1, verbosity=0,
)

xgb_oof  = np.zeros(len(X_train))
xgb_test = np.zeros(len(X_test))

for fold_i, (tr, val) in enumerate(splits):
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_train.iloc[tr], y_train.iloc[tr],
          eval_set=[(X_train.iloc[val], y_train.iloc[val])], verbose=100)
    xgb_oof[val]  = m.predict(X_train.iloc[val])
    xgb_test     += m.predict(X_test) / N_FOLDS
    fold_rmse = np.sqrt(mean_squared_error(y_train.iloc[val], xgb_oof[val]))
    print(f'  Fold {fold_i}  RMSE: {fold_rmse:.4f}  best_iter={m.best_iteration}')

xgb_rmse = np.sqrt(mean_squared_error(y_train, xgb_oof))
print(f'\nXGBoost OOF RMSE: {xgb_rmse:.4f}')

## 6. CatBoost

In [ ]:
cat_oof  = np.zeros(len(X_train))
cat_test = np.zeros(len(X_test))

for fold_i, (tr, val) in enumerate(splits):
    m = CatBoostRegressor(
        iterations=500, learning_rate=0.1, depth=6,
        l2_leaf_reg=3, loss_function='RMSE',
        early_stopping_rounds=30, random_seed=SEED, verbose=100,
    )
    m.fit(Pool(X_train.iloc[tr], y_train.iloc[tr]),
          eval_set=Pool(X_train.iloc[val], y_train.iloc[val]))
    cat_oof[val]  = m.predict(X_train.iloc[val])
    cat_test     += m.predict(X_test) / N_FOLDS
    fold_rmse = np.sqrt(mean_squared_error(y_train.iloc[val], cat_oof[val]))
    print(f'  Fold {fold_i}  RMSE: {fold_rmse:.4f}')

cat_rmse = np.sqrt(mean_squared_error(y_train, cat_oof))
print(f'\nCatBoost OOF RMSE: {cat_rmse:.4f}')

## 7. Ensemble

In [ ]:
oof_mat  = np.column_stack([lgb_oof,  xgb_oof,  cat_oof])
test_mat = np.column_stack([lgb_test, xgb_test, cat_test])

def neg_rmse(w):
    w = np.abs(w) / np.abs(w).sum()
    return np.sqrt(mean_squared_error(y_train, oof_mat @ w))

res = minimize(neg_rmse, [1/3, 1/3, 1/3], method='Nelder-Mead',
               options={'maxiter': 2000, 'xatol': 1e-7})
best_w = np.abs(res.x) / np.abs(res.x).sum()

ensemble_oof  = oof_mat  @ best_w
ensemble_test = test_mat @ best_w
ens_rmse = np.sqrt(mean_squared_error(y_train, ensemble_oof))

print(f'Weights → LGB: {best_w[0]:.3f}  XGB: {best_w[1]:.3f}  CAT: {best_w[2]:.3f}')
print(f'\nOOF RMSE comparison:')
print(f'  LightGBM : {lgb_rmse:.4f}')
print(f'  XGBoost  : {xgb_rmse:.4f}')
print(f'  CatBoost : {cat_rmse:.4f}')
print(f'  Ensemble : {ens_rmse:.4f}')

## 8. Feature Importance

In [ ]:
imp_model = lgb.train(
    {**lgb_params, 'learning_rate': 0.05},
    lgb.Dataset(X_train, y_train),
    num_boost_round=500,
    callbacks=[lgb.log_evaluation(100)]
)

imp = pd.DataFrame({'feature': feature_cols,
                    'gain': imp_model.feature_importance('gain')
                   }).sort_values('gain', ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
imp.head(35).plot.barh(x='feature', y='gain', ax=ax, legend=False, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 35 features by LightGBM gain')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.show()

print(imp.head(15).to_string(index=False))

## 9. Residual Analysis

In [ ]:
residuals = y_train.values - ensemble_oof
idx = np.random.choice(len(y_train), min(8000, len(y_train)), replace=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_train.values[idx], ensemble_oof[idx], alpha=0.25, s=3)
lo, hi = y_train.quantile(0.01), y_train.quantile(0.99)
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_title('Predicted vs Actual TVT')
axes[0].set_xlabel('Actual TVT')
axes[0].set_ylabel('Predicted TVT')

axes[1].hist(residuals, bins=100, color='steelblue', edgecolor='none')
axes[1].axvline(0, color='red', lw=2)
axes[1].set_title(f'Residuals  mean={residuals.mean():.3f}  std={residuals.std():.3f}')
axes[1].set_xlabel('Residual (m)')

# Per-well RMSE
diag = train_fe[[WELL_COL]].copy()
diag['pred'] = ensemble_oof
diag['sq_err'] = (y_train.values - ensemble_oof) ** 2
well_rmse = diag.groupby(WELL_COL)['sq_err'].mean().apply(np.sqrt).sort_values(ascending=False)
axes[2].hist(well_rmse, bins=40, color='tomato', edgecolor='none')
axes[2].axvline(well_rmse.median(), color='black', ls='--', lw=1.5,
                label=f'Median: {well_rmse.median():.2f}m')
axes[2].set_title('Per-well RMSE distribution')
axes[2].set_xlabel('RMSE (m)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 10. Generate Submission

In [ ]:
# Build id column for test set
test_fe['sub_id'] = build_submission_id(test_fe)
test_fe['pred']   = ensemble_test

# Filter to submission rows only
sub_mask = get_submission_mask(test_fe, sample_sub)
print(f'Submission rows matched: {sub_mask.sum()} / {len(sample_sub)}')

submission = test_fe[sub_mask][['sub_id', 'pred']].copy()
submission.columns = ['id', 'tvt']   # match submission format exactly

# Sort to match sample submission order
sub_order = pd.Series(range(len(sample_sub)), index=sample_sub['id'])
submission = submission.set_index('id').loc[sample_sub['id']].reset_index()
submission.columns = ['id', 'tvt']

assert len(submission) == len(sample_sub), 'Row count mismatch!'

out_path = SUBMIT_DIR / 'submission_ensemble.csv'
submission.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(submission.head(10))
print('\nPrediction stats:')
print(submission['tvt'].describe().round(2))

## 11. Sanity Check: Visualise Predictions on Test Wells

In [ ]:
for wid in test[WELL_COL].unique():
    # Also get the full train version of this well for ground truth
    tr_well   = train_fe[train_fe[WELL_COL] == wid].sort_values(DEPTH_COL)
    te_well   = test_fe[test_fe[WELL_COL] == wid].sort_values(DEPTH_COL)

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))

    axes[0].plot(tr_well[DEPTH_COL], tr_well[TARGET_COL], lw=1.5, color='black', label='True TVT')
    axes[0].plot(te_well[DEPTH_COL], te_well['pred'], lw=1.2, color='tomato', ls='--', label='Predicted')
    axes[0].set_title(f'Well {wid} — TVT prediction vs ground truth')
    axes[0].set_xlabel('MD (m)')
    axes[0].set_ylabel('TVT (m)')
    axes[0].legend()

    axes[1].plot(te_well[DEPTH_COL], te_well[GR_COL], lw=0.8, color='darkorange')
    axes[1].set_title(f'Well {wid} — GR log')
    axes[1].set_xlabel('MD (m)')
    axes[1].set_ylabel('GR (API)')

    plt.tight_layout()
    plt.show()